In [1]:
# 데이터 생성 : csv파일을 불러옴

import pandas as pd

df = pd.read_csv('../data/csv/sentiment_data.csv')
df.head()

,sentence,label
0,오늘 전혀 슬퍼요,0
1,공부 진짜 멋져요,1
2,음악 진짜 좋아요,1
3,음악 진짜 추천해요,1
4,밥 매우 재밌어요,1


In [2]:
# 독립, 종속 변수 분리

X = df['sentence']
y = df['label']

y

0      0
1      1
2      1
3      1
4      1
      ..
995    0
996    0
997    1
998    1
999    0
Name: label, Length: 1000, dtype: int64

In [3]:
# 훈련, 테스트 분리

DATA_SIZE = 1000
TRAIN_SIZE = int(DATA_SIZE * 0.8)

X_train, X_test = X[:TRAIN_SIZE], X[TRAIN_SIZE:]
y_train, y_test = y[:TRAIN_SIZE], y[TRAIN_SIZE:]

In [4]:
from konlpy.tag import Okt

def get_preprocessing(sentence):
  okt = Okt()
  result = okt.pos(sentence, stem=True) # 문장을 형태소별로 나눔. 단 원형으로 
  words = [word for word, pos in result]
  return " ".join(words)


print(X_train[0])
print(get_preprocessing(X_train[0]))

# X_train과 X_test의 있는 문장들을 get_preprocessing을 적용

오늘 전혀 슬퍼요
오늘 전혀 슬프다


In [5]:
# 벡터화
from tensorflow.keras import layers, models
import tensorflow as tf

vectorize_layer = layers.TextVectorization(
  max_tokens = 1000,
  output_mode = "multi_hot"
)
vectorize_layer.adapt(X_train.tolist())


In [6]:
# 모델 설계
model = models.Sequential([
  layers.Input((1,), dtype= tf.string),
  vectorize_layer,
  # 히든층
  layers.Dense(64, activation='relu'),
  layers.Dense(32, activation='relu'),
  layers.Dense(1, activation='sigmoid') # 출력층 (결과가 0,1이니까 sigmoid)  
])


In [7]:
# 모델 설정
model.compile(
  optimizer = 'adam',
  loss = 'binary_crossentropy', 
  metrics=['accuracy']
)

In [8]:
# 학습
history = model.fit(
  X_train.values, y_train, epochs=100, verbose=1, validation_split=0.2, 
  batch_size=32
)

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.6672 - loss: 0.6630 - val_accuracy: 0.7812 - val_loss: 0.6287
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8891 - loss: 0.5940 - val_accuracy: 0.9000 - val_loss: 0.5492
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9375 - loss: 0.4922 - val_accuracy: 0.9438 - val_loss: 0.4213
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9766 - loss: 0.3476 - val_accuracy: 0.9937 - val_loss: 0.2601
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9984 - loss: 0.1980 - val_accuracy: 1.0000 - val_loss: 0.1320
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0983 - val_accuracy: 1.0000 - val_loss: 0.0641
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0498 - val_accuracy: 1.0000 - val_loss: 0.0347
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.0277 - val_accuracy: 1.0000 - 

In [9]:
# 평가
_, acc = model.evaluate(X_test.values, y_test)
print(f'정확도 : {acc}')


7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 5.2386e-05  
정확도 : 1.0


In [14]:
# 예측

import numpy as np

# 너무 피곤하고 듣고 있는 음악은 별로지만 기분이 좋아요
# 오늘 날이 좋고 기분이 좋아요
# 듣고 있는 음악이 슬퍼서 우울해요 

sentences = [
	'음악 선곡이 가슴을 울리고 배우들의 몰입감이 엄청나며 연출력이 세계적인 수준이다.',
]

predictions = model.predict(np.array(sentences, dtype=object))
predictions


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step


array([[0.48335978]], dtype=float32)